# VIRCON-Classify — IJIES Continuation: Full Strengthening Run

This notebook **continues from** `VIRCON_IJIES_Reviewer_Revision_Artifacts.zip`. It does not replace the completed reviewer rerun.

It adds the remaining high-value evidence:

1. a **fine-tuned DistilBERT** contemporary baseline under the same fixed-split and event-disjoint protocols;
2. **external HumAID transfer validation** with exact-overlap removal and event-wise reporting;
3. **confidence/abstention (selective prediction)** analysis without claiming human-workflow validation;
4. **event-level domain-shift diagnostics** (lexical Jensen–Shannon divergence and OOV token mass versus event performance);
5. a **two-rater manual triage audit template** plus agreement/sensitivity analysis when annotations are completed;
6. reviewer-facing tables, predictions, checkpoints, hashes, and a final continuation artifact ZIP.

### Important scientific boundaries
- No experiment can guarantee acceptance.
- The existing CrisisBench event-disjoint results remain a **same-corpus event-shift** test.
- HumAID is used as a separate external dataset for transfer evaluation; exact normalized text overlap with CrisisBench is checked and removed.
- Human/practitioner validation is **not fabricated**. The audit analysis runs only after two raters fill the provided template.

### Runtime
For the full transformer run, use a Colab **GPU runtime**. T4/L4/A100 is suitable. The notebook checkpoints every fold, so rerunning can resume completed work.

In [ ]:
# 1. Mount Google Drive and enter the existing VIRCON folder.
from pathlib import Path
import os, sys, json, zipfile, hashlib, platform, subprocess, time, random, math, re, html, warnings, gc

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Google Drive mount skipped or already available:', exc)

WORKDIR = Path('/content/drive/MyDrive/VirconPaper1')
if not WORKDIR.is_dir():
    raise FileNotFoundError(f'Expected Drive folder not found: {WORKDIR}')
os.chdir(WORKDIR)
print('Working directory:', Path.cwd())

TRAIN_PATH = WORKDIR / 'crisis_consolidated_humanitarian_filtered_lang_en_train.tsv'
TEST_PATH = WORKDIR / 'crisis_consolidated_humanitarian_filtered_lang_en_test.tsv'
PRIOR_ZIP = WORKDIR / 'VIRCON_IJIES_Reviewer_Revision_Artifacts.zip'
for p in (TRAIN_PATH, TEST_PATH, PRIOR_ZIP):
    if not p.is_file():
        raise FileNotFoundError(f'Missing required continuation input: {p}')
    print('FOUND:', p.name, p.stat().st_size, 'bytes')


In [ ]:
# 2. Colab-safe dependency check. Do not upgrade NumPy/Pandas/PyTorch.
import importlib.util

for mod, spec in [
    ('transformers', 'transformers>=4.45,<5'),
    ('requests', 'requests>=2.31'),
]:
    if importlib.util.find_spec(mod) is None:
        print('Installing missing package:', spec)
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade-strategy', 'only-if-needed', spec])
    else:
        print(mod, 'already available')

import numpy as np
import pandas as pd
import scipy
import sklearn
import torch
import requests
import joblib
from scipy import sparse
from scipy.spatial.distance import jensenshannon
from scipy.stats import spearmanr
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_recall_fscore_support, log_loss
)
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('Pandas:', pd.__version__)
print('scikit-learn:', sklearn.__version__)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Full continuation requires a GPU. In Colab choose Runtime -> Change runtime type -> GPU, then rerun.')
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# 3. Configuration and reproducibility.
SEED = 20261001
MODEL_ID = 'distilbert-base-uncased'
MAX_LENGTH = 128
TRAIN_BATCH_SIZE = 32
EVAL_BATCH_SIZE = 128
FIXED_EPOCHS = 2
EVENT_EPOCHS = 2
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
GRAD_CLIP = 1.0
RUN_EVENT_DIRECT_TRIAGE = True   # full reviewer-facing run
RUN_EXTERNAL_HUMAID = True
N_BOOTSTRAP = 2000

OUT_ROOT = WORKDIR / 'artifacts_ijies_continuation_strengthening'
TABLE_DIR = OUT_ROOT / 'tables'
REPORT_DIR = OUT_ROOT / 'reports'
MODEL_DIR = OUT_ROOT / 'models'
CHECKPOINT_DIR = OUT_ROOT / 'checkpoints'
for d in (OUT_ROOT, TABLE_DIR, REPORT_DIR, MODEL_DIR, CHECKPOINT_DIR):
    d.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

CONFIG = {
    'seed': SEED, 'model_id': MODEL_ID, 'max_length': MAX_LENGTH,
    'train_batch_size': TRAIN_BATCH_SIZE, 'eval_batch_size': EVAL_BATCH_SIZE,
    'fixed_epochs': FIXED_EPOCHS, 'event_epochs': EVENT_EPOCHS,
    'learning_rate': LEARNING_RATE, 'weight_decay': WEIGHT_DECAY,
    'warmup_ratio': WARMUP_RATIO, 'gradient_clip': GRAD_CLIP,
    'event_direct_triage': RUN_EVENT_DIRECT_TRIAGE,
    'external_humaid': RUN_EXTERNAL_HUMAID, 'n_bootstrap': N_BOOTSTRAP,
    'device_training': 'cuda', 'deployment_benchmark_device': 'cpu',
}
(REPORT_DIR / 'continuation_config.json').write_text(json.dumps(CONFIG, indent=2), encoding='utf-8')
print(json.dumps(CONFIG, indent=2))


In [ ]:
# 4. Restore the completed reviewer-rerun artifact bundle.
PRIOR_EXTRACT = WORKDIR / '_vircon_prior_reviewer_artifacts'
if not PRIOR_EXTRACT.exists():
    PRIOR_EXTRACT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(PRIOR_ZIP, 'r') as z:
        bad = z.testzip()
        if bad is not None:
            raise zipfile.BadZipFile(f'Corrupt member in prior artifact ZIP: {bad}')
        z.extractall(PRIOR_EXTRACT)

PRIOR_ROOT = PRIOR_EXTRACT / 'artifacts_ijies_reviewer_revision'
PRIOR_TABLE = PRIOR_ROOT / 'tables'
required_prior = [
    'fixed_split_predictions.csv',
    'event_disjoint_oof_predictions.csv',
    'event_disjoint_fold_assignment.csv',
    'complete_sample_accounting.csv',
    'triage_manual_audit_template.csv',
]
for name in required_prior:
    p = PRIOR_TABLE / name
    if not p.is_file():
        raise FileNotFoundError(f'Prior artifact missing: {p}')
print('Prior reviewer artifacts restored:', PRIOR_ROOT)


In [ ]:
# 5. Reconstruct the exact prepared CrisisBench records and normalization.
URL_RE = re.compile(r'(?:https?://\S+|www\.\S+)', flags=re.IGNORECASE)
USER_RE = re.compile(r'@[A-Za-z0-9_]+')
HASHTAG_RE = re.compile(r'#([A-Za-z0-9_]+)')
NUMBER_RE = re.compile(r'\b\d+(?:[.,]\d+)?\b')
NON_TOKEN_RE = re.compile(r"[^a-z0-9_\s']+")
SPACE_RE = re.compile(r'\s+')

REQUEST_SET = {'requests_or_needs'}
RESPONSE_SET = {'response_efforts', 'donation_and_volunteering'}
TRIAGE_LABELS = ['other', 'request', 'response']

def normalize_text(value):
    if value is None:
        return ''
    try:
        if bool(pd.isna(value)):
            return ''
    except Exception:
        pass
    text = html.unescape(str(value)).lower()
    text = URL_RE.sub(' urltoken ', text)
    text = USER_RE.sub(' usertoken ', text)
    text = HASHTAG_RE.sub(r' \1 ', text)
    text = NUMBER_RE.sub(' numtoken ', text)
    text = NON_TOKEN_RE.sub(' ', text)
    return SPACE_RE.sub(' ', text).strip()

def triage_map(label):
    if label in REQUEST_SET: return 'request'
    if label in RESPONSE_SET: return 'response'
    return 'other'

def read_tsv(path, split_name):
    df = pd.read_csv(path, sep='\t')
    for col in ('text','class_label','event'):
        if col not in df.columns:
            raise KeyError(f'{split_name} missing required column {col}; columns={list(df.columns)}')
    df = df.copy()
    df['raw_text'] = df['text'].fillna('').astype(str)
    df['clean_text'] = df['raw_text'].map(normalize_text)
    df['class_label'] = df['class_label'].astype(str).str.strip()
    mask = df['class_label'].ne('') & df['clean_text'].ne('')
    df = df.loc[mask].reset_index(drop=True)
    df['source_split'] = split_name
    df['source_row'] = np.arange(len(df), dtype=np.int64)
    return df

train_df = read_tsv(TRAIN_PATH, 'train')
test_df = read_tsv(TEST_PATH, 'test')
combined = pd.concat([train_df, test_df], ignore_index=True, sort=False)
combined['global_row_id'] = np.arange(len(combined), dtype=np.int64)
combined['label_11'] = combined['class_label'].astype(str)
combined['label_triage'] = combined['label_11'].map(triage_map)
LABELS = sorted(train_df['class_label'].astype(str).unique().tolist())

prior_fixed = pd.read_csv(PRIOR_TABLE / 'fixed_split_predictions.csv')
prior_event = pd.read_csv(PRIOR_TABLE / 'event_disjoint_oof_predictions.csv')
prior_event_assign = pd.read_csv(PRIOR_TABLE / 'event_disjoint_fold_assignment.csv')

if len(train_df) != 61164 or len(test_df) != 17356:
    raise RuntimeError(f'Prepared row counts changed: train={len(train_df)}, test={len(test_df)}')
if len(combined) != 78520:
    raise RuntimeError('Prepared combined population must remain 78,520 records.')

strict_ids = prior_event['global_row_id'].astype(int).to_numpy()
strict_df = combined.set_index('global_row_id').loc[strict_ids].reset_index()
strict_df = strict_df.merge(
    prior_event_assign[['global_row_id','canonical_event','fold']],
    on='global_row_id', how='left', validate='one_to_one'
)
if strict_df['fold'].isna().any():
    raise RuntimeError('Strict population did not match saved event-disjoint fold assignments.')

print('Prepared train/test:', len(train_df), len(test_df))
print('Strict event population:', len(strict_df), 'events:', strict_df['canonical_event'].nunique())
print('Labels:', LABELS)


## Fine-tuned contemporary transformer

The previous rerun used a **frozen MiniLM embedding + logistic regression** baseline. This continuation adds **end-to-end supervised DistilBERT fine-tuning**. Hyperparameters are fixed before test evaluation; no official test labels are used for model selection.

In [ ]:
# 6. Transformer helpers: fixed epochs, class-weighted cross-entropy, checkpointable predictions.
DEVICE = torch.device('cuda')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = list(map(str, texts))
        self.labels = np.asarray(labels, dtype=np.int64)
        self.tokenizer = tokenizer
        self.max_length = int(max_length)
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx], truncation=True, padding='max_length',
            max_length=self.max_length, return_tensors='pt'
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item['labels'] = torch.tensor(int(self.labels[idx]), dtype=torch.long)
        return item

def make_label_maps(label_names):
    label_names = list(label_names)
    return {x:i for i,x in enumerate(label_names)}, {i:x for i,x in enumerate(label_names)}

def class_weights_from_y(y_int, n_classes):
    present = np.unique(y_int)
    weights = np.ones(n_classes, dtype=np.float32)
    vals = compute_class_weight(class_weight='balanced', classes=present, y=y_int)
    for cls, w in zip(present, vals): weights[int(cls)] = float(w)
    return torch.tensor(weights, dtype=torch.float32, device=DEVICE)

def train_transformer_fixed_epochs(train_texts, train_labels_str, label_names, epochs, run_name):
    label2id, id2label = make_label_maps(label_names)
    y = np.asarray([label2id[x] for x in train_labels_str], dtype=np.int64)
    ds = TextDataset(train_texts, y, tokenizer, MAX_LENGTH)
    gen = torch.Generator(); gen.manual_seed(SEED)
    loader = DataLoader(ds, batch_size=TRAIN_BATCH_SIZE, shuffle=True, generator=gen, num_workers=2, pin_memory=True)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID, num_labels=len(label_names), label2id=label2id, id2label=id2label
    ).to(DEVICE)
    weights = class_weights_from_y(y, len(label_names))
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    total_steps = max(1, len(loader) * int(epochs))
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(WARMUP_RATIO * total_steps), num_training_steps=total_steps
    )
    scaler = torch.amp.GradScaler('cuda', enabled=True)
    history=[]
    for epoch in range(1, int(epochs)+1):
        model.train(); losses=[]; t0=time.time()
        for batch in loader:
            labels = batch.pop('labels').to(DEVICE, non_blocking=True)
            batch = {k:v.to(DEVICE, non_blocking=True) for k,v in batch.items()}
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', enabled=True):
                logits = model(**batch).logits
                loss = torch.nn.functional.cross_entropy(logits, labels, weight=weights)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer); scaler.update(); scheduler.step()
            losses.append(float(loss.detach().cpu()))
        row={'run_name':run_name,'epoch':epoch,'mean_train_loss':float(np.mean(losses)),'seconds':float(time.time()-t0)}
        history.append(row); print(row)
    return model, pd.DataFrame(history), label2id, id2label

@torch.no_grad()
def predict_transformer(model, texts, id2label, return_prob=True, device=DEVICE):
    dummy = np.zeros(len(texts), dtype=np.int64)
    ds = TextDataset(texts, dummy, tokenizer, MAX_LENGTH)
    loader = DataLoader(ds, batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=(device.type=='cuda'))
    model = model.to(device); model.eval()
    preds=[]; probs=[]
    for batch in loader:
        batch.pop('labels', None)
        batch = {k:v.to(device, non_blocking=(device.type=='cuda')) for k,v in batch.items()}
        logits = model(**batch).logits
        p = torch.softmax(logits, dim=-1)
        idx = torch.argmax(p, dim=-1).detach().cpu().numpy()
        preds.extend([id2label[int(i)] for i in idx])
        if return_prob: probs.append(p.detach().cpu().numpy())
    prob_arr = np.vstack(probs) if return_prob else None
    return np.asarray(preds, dtype=object), prob_arr

def metric_row(y_true, y_pred, labels):
    labels=list(labels)
    return {
        'n': int(len(y_true)),
        'accuracy': float(accuracy_score(y_true,y_pred)),
        'balanced_accuracy': float(balanced_accuracy_score(y_true,y_pred)),
        'macro_f1': float(f1_score(y_true,y_pred,labels=labels,average='macro',zero_division=0)),
        'weighted_f1': float(f1_score(y_true,y_pred,labels=labels,average='weighted',zero_division=0)),
    }

def save_class_report(y_true, y_pred, labels, path):
    rep=classification_report(y_true,y_pred,labels=list(labels),output_dict=True,zero_division=0)
    pd.DataFrame(rep).T.reset_index().rename(columns={'index':'label'}).to_csv(path,index=False)


In [ ]:
# 7. Official fixed-split fine-tuned DistilBERT: 11-class and direct triage.
fixed_rows=[]
fixed_pred_path = TABLE_DIR / 'distilbert_fixed_split_predictions.csv'
fixed_model_11_dir = MODEL_DIR / 'distilbert_fixed_11class'
fixed_model_tri_dir = MODEL_DIR / 'distilbert_fixed_direct_triage'

if fixed_pred_path.is_file():
    print('Reusing completed fixed-split DistilBERT outputs.')
    distil_fixed_pred = pd.read_csv(fixed_pred_path)
else:
    # 11 class
    model11, hist11, l2i11, i2l11 = train_transformer_fixed_epochs(
        train_df['clean_text'].tolist(), train_df['class_label'].tolist(), LABELS, FIXED_EPOCHS, 'fixed_11class'
    )
    pred11, prob11 = predict_transformer(model11, test_df['clean_text'].tolist(), i2l11)
    hist11.to_csv(TABLE_DIR/'distilbert_fixed_11class_training_history.csv', index=False)
    save_class_report(test_df['class_label'].to_numpy(), pred11, LABELS, TABLE_DIR/'distilbert_fixed_11class_per_class.csv')
    fixed_model_11_dir.mkdir(parents=True, exist_ok=True)
    model11.save_pretrained(fixed_model_11_dir); tokenizer.save_pretrained(fixed_model_11_dir)
    np.save(TABLE_DIR/'distilbert_fixed_11class_probabilities.npy', prob11)
    del model11; gc.collect(); torch.cuda.empty_cache()

    # Direct triage
    modeltri, histtri, l2it, i2lt = train_transformer_fixed_epochs(
        train_df['clean_text'].tolist(), train_df['class_label'].map(triage_map).tolist(), TRIAGE_LABELS, FIXED_EPOCHS, 'fixed_direct_triage'
    )
    predtri, probtri = predict_transformer(modeltri, test_df['clean_text'].tolist(), i2lt)
    histtri.to_csv(TABLE_DIR/'distilbert_fixed_direct_triage_training_history.csv', index=False)
    save_class_report(test_df['class_label'].map(triage_map).to_numpy(), predtri, TRIAGE_LABELS, TABLE_DIR/'distilbert_fixed_direct_triage_per_class.csv')
    fixed_model_tri_dir.mkdir(parents=True, exist_ok=True)
    modeltri.save_pretrained(fixed_model_tri_dir); tokenizer.save_pretrained(fixed_model_tri_dir)
    np.save(TABLE_DIR/'distilbert_fixed_direct_triage_probabilities.npy', probtri)
    del modeltri; gc.collect(); torch.cuda.empty_cache()

    distil_fixed_pred = pd.DataFrame({
        'global_row_id': np.arange(len(train_df),len(train_df)+len(test_df),dtype=np.int64),
        'label_11': test_df['class_label'].to_numpy(),
        'label_triage': test_df['class_label'].map(triage_map).to_numpy(),
        'pred_11class__distilbert_ft': pred11,
        'pred_direct_triage__distilbert_ft': predtri,
    })
    distil_fixed_pred.to_csv(fixed_pred_path,index=False)

fixed_rows.append({'protocol':'official_fixed_split','task':'11class','model':'distilbert_ft',**metric_row(distil_fixed_pred['label_11'],distil_fixed_pred['pred_11class__distilbert_ft'],LABELS)})
fixed_rows.append({'protocol':'official_fixed_split','task':'direct_triage','model':'distilbert_ft',**metric_row(distil_fixed_pred['label_triage'],distil_fixed_pred['pred_direct_triage__distilbert_ft'],TRIAGE_LABELS)})
pd.DataFrame(fixed_rows).to_csv(TABLE_DIR/'distilbert_fixed_split_summary.csv',index=False)
print(pd.DataFrame(fixed_rows))


In [ ]:
# 8. Exact saved 5-fold event-disjoint protocol for fine-tuned DistilBERT.
event_fold_summaries=[]
event_oof = strict_df[['global_row_id','canonical_event','fold','label_11','label_triage']].copy()
event_oof['pred_11class__distilbert_ft'] = None
if RUN_EVENT_DIRECT_TRIAGE:
    event_oof['pred_direct_triage__distilbert_ft'] = None

for fold in sorted(strict_df['fold'].astype(int).unique()):
    fold_ckpt = CHECKPOINT_DIR / f'distilbert_event_fold_{fold}.csv'
    if fold_ckpt.is_file():
        print('Reusing fold checkpoint', fold)
        ck = pd.read_csv(fold_ckpt)
        pos = event_oof['fold'].astype(int).eq(fold)
        ck = ck.set_index('global_row_id')
        ids = event_oof.loc[pos,'global_row_id'].astype(int)
        event_oof.loc[pos,'pred_11class__distilbert_ft'] = ids.map(ck['pred_11class__distilbert_ft']).to_numpy()
        if RUN_EVENT_DIRECT_TRIAGE and 'pred_direct_triage__distilbert_ft' in ck.columns:
            event_oof.loc[pos,'pred_direct_triage__distilbert_ft'] = ids.map(ck['pred_direct_triage__distilbert_ft']).to_numpy()
        continue

    train_mask = strict_df['fold'].astype(int).ne(fold).to_numpy()
    test_mask = ~train_mask
    tr = strict_df.loc[train_mask]
    te = strict_df.loc[test_mask]
    print(f'\nEVENT FOLD {fold}: train={len(tr):,} test={len(te):,} held events={te.canonical_event.nunique()}')

    model11, hist, l2i, i2l = train_transformer_fixed_epochs(
        tr['clean_text'].tolist(), tr['label_11'].tolist(), LABELS, EVENT_EPOCHS, f'event_fold_{fold}_11class'
    )
    p11, _ = predict_transformer(model11, te['clean_text'].tolist(), i2l, return_prob=False)
    hist.to_csv(TABLE_DIR/f'distilbert_event_fold_{fold}_11class_history.csv',index=False)
    del model11; gc.collect(); torch.cuda.empty_cache()

    ck = te[['global_row_id','canonical_event','fold','label_11','label_triage']].copy()
    ck['pred_11class__distilbert_ft'] = p11

    if RUN_EVENT_DIRECT_TRIAGE:
        modelt, histt, l2it, i2lt = train_transformer_fixed_epochs(
            tr['clean_text'].tolist(), tr['label_triage'].tolist(), TRIAGE_LABELS, EVENT_EPOCHS, f'event_fold_{fold}_direct_triage'
        )
        pt, _ = predict_transformer(modelt, te['clean_text'].tolist(), i2lt, return_prob=False)
        histt.to_csv(TABLE_DIR/f'distilbert_event_fold_{fold}_triage_history.csv',index=False)
        ck['pred_direct_triage__distilbert_ft'] = pt
        del modelt; gc.collect(); torch.cuda.empty_cache()

    ck.to_csv(fold_ckpt,index=False)
    pos = event_oof['fold'].astype(int).eq(fold)
    mapper11 = ck.set_index('global_row_id')['pred_11class__distilbert_ft']
    event_oof.loc[pos,'pred_11class__distilbert_ft'] = event_oof.loc[pos,'global_row_id'].map(mapper11).to_numpy()
    if RUN_EVENT_DIRECT_TRIAGE:
        mappert = ck.set_index('global_row_id')['pred_direct_triage__distilbert_ft']
        event_oof.loc[pos,'pred_direct_triage__distilbert_ft'] = event_oof.loc[pos,'global_row_id'].map(mappert).to_numpy()

if event_oof['pred_11class__distilbert_ft'].isna().any():
    raise RuntimeError('Missing 11-class OOF predictions after event folds.')
if RUN_EVENT_DIRECT_TRIAGE and event_oof['pred_direct_triage__distilbert_ft'].isna().any():
    raise RuntimeError('Missing direct-triage OOF predictions after event folds.')

event_oof.to_csv(TABLE_DIR/'distilbert_event_disjoint_oof_predictions.csv',index=False)
rows=[{'protocol':'event_disjoint_same_corpus','task':'11class','model':'distilbert_ft',**metric_row(event_oof['label_11'],event_oof['pred_11class__distilbert_ft'],LABELS)}]
if RUN_EVENT_DIRECT_TRIAGE:
    rows.append({'protocol':'event_disjoint_same_corpus','task':'direct_triage','model':'distilbert_ft',**metric_row(event_oof['label_triage'],event_oof['pred_direct_triage__distilbert_ft'],TRIAGE_LABELS)})
pd.DataFrame(rows).to_csv(TABLE_DIR/'distilbert_event_disjoint_summary.csv',index=False)
save_class_report(event_oof['label_11'],event_oof['pred_11class__distilbert_ft'],LABELS,TABLE_DIR/'distilbert_event_disjoint_11class_per_class.csv')
if RUN_EVENT_DIRECT_TRIAGE:
    save_class_report(event_oof['label_triage'],event_oof['pred_direct_triage__distilbert_ft'],TRIAGE_LABELS,TABLE_DIR/'distilbert_event_disjoint_triage_per_class.csv')
print(pd.DataFrame(rows))


In [ ]:
# 9. Primary effect estimates: DistilBERT versus prior sparse and frozen-MiniLM models.
def confusion_fixed(y_true, y_pred, labels):
    return confusion_matrix(y_true,y_pred,labels=list(labels)).astype(float)

def macro_f1_from_cm(cm):
    tp=np.diag(cm); fp=cm.sum(0)-tp; fn=cm.sum(1)-tp
    den=2*tp+fp+fn
    vals=np.divide(2*tp,den,out=np.zeros_like(tp,dtype=float),where=den>0)
    return float(vals.mean())

def stratified_paired_bootstrap(y_true, pred_a, pred_b, labels, n_boot, seed):
    y_true=np.asarray(y_true,dtype=object); a=np.asarray(pred_a,dtype=object); b=np.asarray(pred_b,dtype=object)
    labels=list(labels); rng=np.random.default_rng(seed)
    idx_by=[np.flatnonzero(y_true==lab) for lab in labels]
    observed=macro_f1_from_cm(confusion_fixed(y_true,a,labels))-macro_f1_from_cm(confusion_fixed(y_true,b,labels))
    vals=[]
    for _ in range(n_boot):
        idx=np.concatenate([rng.choice(ix,size=len(ix),replace=True) for ix in idx_by if len(ix)])
        vals.append(macro_f1_from_cm(confusion_fixed(y_true[idx],a[idx],labels))-macro_f1_from_cm(confusion_fixed(y_true[idx],b[idx],labels)))
    lo,hi=np.quantile(vals,[.025,.975])
    return observed,float(lo),float(hi)

def event_cluster_bootstrap(y_true,pred_a,pred_b,groups,labels,n_boot,seed):
    y_true=np.asarray(y_true,dtype=object); a=np.asarray(pred_a,dtype=object); b=np.asarray(pred_b,dtype=object); g=np.asarray(groups,dtype=object)
    events=sorted(np.unique(g).tolist()); labels=list(labels)
    cms_a=[]; cms_b=[]
    for ev in events:
        m=g==ev; cms_a.append(confusion_fixed(y_true[m],a[m],labels)); cms_b.append(confusion_fixed(y_true[m],b[m],labels))
    cms_a=np.stack(cms_a); cms_b=np.stack(cms_b); rng=np.random.default_rng(seed)
    observed=macro_f1_from_cm(cms_a.sum(0))-macro_f1_from_cm(cms_b.sum(0)); vals=[]
    for _ in range(n_boot):
        ids=rng.integers(0,len(events),size=len(events))
        vals.append(macro_f1_from_cm(cms_a[ids].sum(0))-macro_f1_from_cm(cms_b[ids].sum(0)))
    lo,hi=np.quantile(vals,[.025,.975])
    return observed,float(lo),float(hi)

comparisons=[]
prior_fixed_idx = prior_fixed.set_index('global_row_id')
new_fixed_idx = distil_fixed_pred.set_index('global_row_id')
merged_fixed = prior_fixed_idx.join(new_fixed_idx[['pred_11class__distilbert_ft','pred_direct_triage__distilbert_ft']],how='inner')
for task,true_col,new_col,labels,prefix in [
    ('11class','label_11','pred_11class__distilbert_ft',LABELS,'pred_11class__'),
    ('direct_triage','label_triage','pred_direct_triage__distilbert_ft',TRIAGE_LABELS,'pred_direct_triage__')]:
    for base in ['tfidf_svc','tfidf_lr','hash_lr','minilm_lr']:
        old_col=prefix+base
        eff,lo,hi=stratified_paired_bootstrap(merged_fixed[true_col],merged_fixed[new_col],merged_fixed[old_col],labels,N_BOOTSTRAP,SEED+len(comparisons))
        comparisons.append({'protocol':'official_fixed_split','task':task,'model_a':'distilbert_ft','model_b':base,'effect_macro_f1_a_minus_b':eff,'ci_2_5':lo,'ci_97_5':hi,'resampling_unit':'message_within_true_class_strata','n_resamples':N_BOOTSTRAP})

prior_event_idx=prior_event.set_index('global_row_id')
new_event_idx=event_oof.set_index('global_row_id')
merged_event=prior_event_idx.join(new_event_idx[['pred_11class__distilbert_ft'] + (['pred_direct_triage__distilbert_ft'] if RUN_EVENT_DIRECT_TRIAGE else [])],how='inner')
for task,true_col,new_col,labels,prefix in [
    ('11class','label_11','pred_11class__distilbert_ft',LABELS,'pred_11class__'),
    ('direct_triage','label_triage','pred_direct_triage__distilbert_ft',TRIAGE_LABELS,'pred_direct_triage__')]:
    if task=='direct_triage' and not RUN_EVENT_DIRECT_TRIAGE: continue
    for base in ['tfidf_svc','tfidf_lr','hash_lr','minilm_lr']:
        old_col=prefix+base
        eff,lo,hi=event_cluster_bootstrap(merged_event[true_col],merged_event[new_col],merged_event[old_col],merged_event['canonical_event'],labels,N_BOOTSTRAP,SEED+100+len(comparisons))
        comparisons.append({'protocol':'event_disjoint_same_corpus','task':task,'model_a':'distilbert_ft','model_b':base,'effect_macro_f1_a_minus_b':eff,'ci_2_5':lo,'ci_97_5':hi,'resampling_unit':'canonical_event_cluster','n_resamples':N_BOOTSTRAP})

pd.DataFrame(comparisons).to_csv(TABLE_DIR/'distilbert_primary_effect_estimates_and_cis.csv',index=False)
print(pd.DataFrame(comparisons).to_string(index=False))


## External validation on HumAID

HumAID is a separately released, human-annotated disaster-event dataset with 19 events. The notebook downloads only the published **test split** for each event, checks for exact normalized-text overlap against CrisisBench, removes overlaps, and reports:

- external **three-way triage** transfer on the full retained HumAID test population;
- external **shared-ontology multi-class** transfer on labels that have an explicit one-to-one counterpart in the CrisisBench label set;
- pooled and event-wise metrics.

This is external validation, but claims should remain specific to the harmonized tasks and dataset.

In [ ]:
# 10. Download and harmonize HumAID test splits from the official QCRI Hugging Face dataset.
HUMAID_EVENTS = [
'hurricane_florence_2018','kaikoura_earthquake_2016','kerala_floods_2018','hurricane_harvey_2017',
'hurricane_maria_2017','midwestern_us_floods_2019','puebla_mexico_earthquake_2017','maryland_floods_2018',
'hurricane_irma_2017','ecuador_earthquake_2016','cyclone_idai_2019','canada_wildfires_2016',
'italy_earthquake_aug_2016','greece_wildfires_2018','hurricane_dorian_2019','california_wildfires_2018',
'pakistan_earthquake_2019','hurricane_matthew_2016','srilanka_floods_2017']

HUMAID_CACHE = WORKDIR / '_humaid_external_cache'; HUMAID_CACHE.mkdir(exist_ok=True)

def load_json_flexible(path):
    txt=Path(path).read_text(encoding='utf-8').strip()
    if not txt: return []
    try:
        obj=json.loads(txt)
        if isinstance(obj,list): return obj
        if isinstance(obj,dict):
            for key in ('data','records','examples'):
                if key in obj and isinstance(obj[key],list): return obj[key]
            return [obj]
    except json.JSONDecodeError:
        return [json.loads(line) for line in txt.splitlines() if line.strip()]

def norm_humaid_label(x):
    x=str(x).strip().lower(); x=re.sub(r'[^a-z0-9]+','_',x).strip('_')
    aliases={
        'caution_and_advice':'caution_and_advice',
        'displaced_people_and_evacuations':'displaced_and_evacuations',
        'displaced_and_evacuations':'displaced_and_evacuations',
        'infrastructure_and_utility_damage':'infrastructure_and_utilities_damage',
        'infrastructure_and_utilities_damage':'infrastructure_and_utilities_damage',
        'injured_or_dead_people':'injured_or_dead_people',
        'missing_or_found_people':'missing_and_found_people',
        'missing_and_found_people':'missing_and_found_people',
        'not_humanitarian':'not_humanitarian',
        'requests_or_urgent_needs':'requests_or_needs',
        'requests_or_needs':'requests_or_needs',
        'rescue_volunteering_or_donation_effort':'donation_and_volunteering',
        'rescue_volunteering_or_donation_efforts':'donation_and_volunteering',
        'sympathy_and_support':'sympathy_and_support',
        'other_relevant_information':'other_relevant_information',
        'dont_know_cant_judge':'dont_know_cant_judge',
        'dont_know_cant_judge_':'dont_know_cant_judge',
    }
    return aliases.get(x,x)

def external_triage(label):
    lab=norm_humaid_label(label)
    if lab=='requests_or_needs': return 'request'
    if lab=='donation_and_volunteering': return 'response'
    return 'other'

if RUN_EXTERNAL_HUMAID:
    external_rows=[]
    for event in HUMAID_EVENTS:
        local=HUMAID_CACHE/f'{event}_test.json'
        if not local.is_file():
            url=f'https://huggingface.co/datasets/QCRI/HumAID-events/resolve/main/{event}/test.json'
            r=requests.get(url,timeout=120); r.raise_for_status(); local.write_bytes(r.content)
        records=load_json_flexible(local)
        for rec in records:
            text_val=rec.get('tweet_text', rec.get('text',''))
            label_val=rec.get('class_label', rec.get('label',''))
            external_rows.append({'external_event':event,'raw_text':str(text_val),'external_label_raw':str(label_val)})
    huma=pd.DataFrame(external_rows)
    huma['clean_text']=huma['raw_text'].map(normalize_text)
    huma['label_harmonized']=huma['external_label_raw'].map(norm_humaid_label)
    huma['label_triage']=huma['external_label_raw'].map(external_triage)
    huma=huma[huma['clean_text'].ne('')].reset_index(drop=True)

    crisis_all_text=set(combined['clean_text'].astype(str))
    huma['exact_overlap_crisisbench']=huma['clean_text'].isin(crisis_all_text)
    overlap_summary=huma.groupby('external_event',as_index=False)['exact_overlap_crisisbench'].agg(['sum','count']).reset_index()
    overlap_summary.to_csv(TABLE_DIR/'humaid_exact_overlap_by_event.csv',index=False)
    huma_clean=huma.loc[~huma['exact_overlap_crisisbench']].copy().reset_index(drop=True)
    huma.to_csv(TABLE_DIR/'humaid_external_test_all_with_overlap_flag.csv',index=False)
    huma_clean.to_csv(TABLE_DIR/'humaid_external_test_overlap_removed.csv',index=False)
    pd.DataFrame({'label':huma_clean['label_harmonized'].value_counts().index,'rows':huma_clean['label_harmonized'].value_counts().values}).to_csv(TABLE_DIR/'humaid_harmonized_label_inventory.csv',index=False)
    print('HumAID test rows downloaded:',len(huma),'after exact-overlap removal:',len(huma_clean),'exact overlaps:',int(huma['exact_overlap_crisisbench'].sum()))
    print(huma_clean['label_harmonized'].value_counts())


In [ ]:
# 11. External transfer: sparse baselines and the fixed-split fine-tuned DistilBERT models.
def build_tfidf():
    return TfidfVectorizer(ngram_range=(1,2),min_df=2,max_df=0.95,max_features=50000,lowercase=False,stop_words='english',token_pattern=r'(?u)\b[a-z][a-z0-9_]{1,}\b',norm='l2',dtype=np.float64)

if RUN_EXTERNAL_HUMAID:
    ext_rows=[]
    # Train sparse 11-class + triage on CrisisBench official training only.
    vec=build_tfidf(); Xtr=vec.fit_transform(train_df['clean_text']); Xext=vec.transform(huma_clean['clean_text'])
    svc11=LinearSVC(C=1.0,class_weight='balanced',max_iter=40000,random_state=SEED).fit(Xtr,train_df['class_label'])
    lr11=LogisticRegression(C=1.0,class_weight='balanced',solver='lbfgs',max_iter=30000,random_state=SEED).fit(Xtr,train_df['class_label'])
    svct=LinearSVC(C=1.0,class_weight='balanced',max_iter=40000,random_state=SEED).fit(Xtr,train_df['class_label'].map(triage_map))
    lrt=LogisticRegression(C=1.0,class_weight='balanced',solver='lbfgs',max_iter=30000,random_state=SEED).fit(Xtr,train_df['class_label'].map(triage_map))

    # Reload fine-tuned fixed models and predict external text.
    tok_ext=AutoTokenizer.from_pretrained(fixed_model_11_dir,use_fast=True)
    tokenizer = tok_ext
    d11=AutoModelForSequenceClassification.from_pretrained(fixed_model_11_dir).to(DEVICE)
    i2l11={int(k):v for k,v in d11.config.id2label.items()}
    p11_ext,_=predict_transformer(d11,huma_clean['clean_text'].tolist(),i2l11,return_prob=False)
    del d11; gc.collect(); torch.cuda.empty_cache()
    dt=AutoModelForSequenceClassification.from_pretrained(fixed_model_tri_dir).to(DEVICE)
    i2lt={int(k):v for k,v in dt.config.id2label.items()}
    pt_ext,pt_prob=predict_transformer(dt,huma_clean['clean_text'].tolist(),i2lt,return_prob=True)
    del dt; gc.collect(); torch.cuda.empty_cache()

    huma_pred=huma_clean[['external_event','external_label_raw','label_harmonized','label_triage','raw_text','clean_text']].copy()
    huma_pred['pred_11class__tfidf_svc']=svc11.predict(Xext)
    huma_pred['pred_11class__tfidf_lr']=lr11.predict(Xext)
    huma_pred['pred_11class__distilbert_ft']=p11_ext
    huma_pred['pred_triage__tfidf_svc']=svct.predict(Xext)
    huma_pred['pred_triage__tfidf_lr']=lrt.predict(Xext)
    huma_pred['pred_triage__distilbert_ft']=pt_ext
    huma_pred.to_csv(TABLE_DIR/'humaid_external_predictions.csv',index=False)
    np.save(TABLE_DIR/'humaid_external_distilbert_triage_probabilities.npy',pt_prob)

    # Full external triage evaluation.
    for model,col in [('tfidf_svc','pred_triage__tfidf_svc'),('tfidf_lr','pred_triage__tfidf_lr'),('distilbert_ft','pred_triage__distilbert_ft')]:
        ext_rows.append({'external_task':'triage_3class','model':model,**metric_row(huma_pred['label_triage'],huma_pred[col],TRIAGE_LABELS)})

    # Shared explicit ontology: only labels with one-to-one CrisisBench counterparts.
    shared_labels=['caution_and_advice','displaced_and_evacuations','infrastructure_and_utilities_damage','injured_or_dead_people','missing_and_found_people','not_humanitarian','requests_or_needs','donation_and_volunteering','sympathy_and_support']
    shared=huma_pred[huma_pred['label_harmonized'].isin(shared_labels)].copy()
    for model,col in [('tfidf_svc','pred_11class__tfidf_svc'),('tfidf_lr','pred_11class__tfidf_lr'),('distilbert_ft','pred_11class__distilbert_ft')]:
        ext_rows.append({'external_task':'shared_9class','model':model,**metric_row(shared['label_harmonized'],shared[col],shared_labels),'unsupported_prediction_rate':float((~shared[col].isin(shared_labels)).mean())})

    ext_summary=pd.DataFrame(ext_rows); ext_summary.to_csv(TABLE_DIR/'humaid_external_transfer_summary.csv',index=False)
    print(ext_summary.to_string(index=False))

    # Event-wise external metrics.
    evrows=[]
    for event,g in huma_pred.groupby('external_event'):
        for model,col in [('tfidf_svc','pred_triage__tfidf_svc'),('tfidf_lr','pred_triage__tfidf_lr'),('distilbert_ft','pred_triage__distilbert_ft')]:
            evrows.append({'external_event':event,'task':'triage_3class','model':model,**metric_row(g['label_triage'],g[col],TRIAGE_LABELS)})
        gs=g[g['label_harmonized'].isin(shared_labels)]
        if len(gs):
            for model,col in [('tfidf_svc','pred_11class__tfidf_svc'),('tfidf_lr','pred_11class__tfidf_lr'),('distilbert_ft','pred_11class__distilbert_ft')]:
                evrows.append({'external_event':event,'task':'shared_9class','model':model,**metric_row(gs['label_harmonized'],gs[col],shared_labels)})
    pd.DataFrame(evrows).to_csv(TABLE_DIR/'humaid_external_eventwise_metrics.csv',index=False)


In [ ]:
# 12. Selective prediction / abstention analysis on official test and external HumAID.
def selective_curve(y_true, pred, probs, classes, labels, model, dataset):
    conf=probs.max(axis=1)
    rows=[]
    for threshold in np.arange(0.34,0.951,0.05):
        accepted=conf>=threshold
        if accepted.sum()==0: continue
        m=metric_row(np.asarray(y_true)[accepted],np.asarray(pred)[accepted],labels)
        req_mask=np.asarray(y_true)[accepted]=='request'
        req_recall=float(np.mean(np.asarray(pred)[accepted][req_mask]=='request')) if req_mask.any() else np.nan
        rows.append({'dataset':dataset,'model':model,'threshold':float(threshold),'coverage':float(accepted.mean()),'abstention_rate':float(1-accepted.mean()),'accepted_n':int(accepted.sum()),'request_recall_within_accepted':req_recall,**m})
    return rows

sel_rows=[]
# DistilBERT official-test probabilities were saved in section 7.
fixed_tri_prob=np.load(TABLE_DIR/'distilbert_fixed_direct_triage_probabilities.npy')
# Class order is saved in the model config.
_tmp=AutoModelForSequenceClassification.from_pretrained(fixed_model_tri_dir)
tri_class_order=[_tmp.config.id2label[i] for i in range(_tmp.config.num_labels)]; del _tmp
sel_rows += selective_curve(distil_fixed_pred['label_triage'].to_numpy(),distil_fixed_pred['pred_direct_triage__distilbert_ft'].to_numpy(),fixed_tri_prob,tri_class_order,TRIAGE_LABELS,'distilbert_ft','crisisbench_official_test')

# Fresh TF-IDF LR probability baseline on official test.
vec_sel=build_tfidf(); Xtr_sel=vec_sel.fit_transform(train_df['clean_text']); Xte_sel=vec_sel.transform(test_df['clean_text'])
lr_sel=LogisticRegression(C=1.0,class_weight='balanced',solver='lbfgs',max_iter=30000,random_state=SEED).fit(Xtr_sel,train_df['class_label'].map(triage_map))
pred_lr_sel=lr_sel.predict(Xte_sel); prob_lr_sel=lr_sel.predict_proba(Xte_sel)
sel_rows += selective_curve(test_df['class_label'].map(triage_map).to_numpy(),pred_lr_sel,prob_lr_sel,lr_sel.classes_,TRIAGE_LABELS,'tfidf_lr','crisisbench_official_test')

if RUN_EXTERNAL_HUMAID:
    prob_ext=np.load(TABLE_DIR/'humaid_external_distilbert_triage_probabilities.npy')
    hp=pd.read_csv(TABLE_DIR/'humaid_external_predictions.csv')
    sel_rows += selective_curve(hp['label_triage'].to_numpy(),hp['pred_triage__distilbert_ft'].to_numpy(),prob_ext,tri_class_order,TRIAGE_LABELS,'distilbert_ft','humaid_external')
    Xext_sel=vec_sel.transform(hp['clean_text']); pred=lr_sel.predict(Xext_sel); prob=lr_sel.predict_proba(Xext_sel)
    sel_rows += selective_curve(hp['label_triage'].to_numpy(),pred,prob,lr_sel.classes_,TRIAGE_LABELS,'tfidf_lr','humaid_external')

pd.DataFrame(sel_rows).to_csv(TABLE_DIR/'selective_prediction_abstention_curves.csv',index=False)
print(pd.DataFrame(sel_rows).head(20).to_string(index=False))


In [ ]:
# 13. Event-level lexical shift diagnostic versus event-disjoint performance.
# Descriptive analysis only; it is not used to tune models or folds.
from collections import Counter

def tokenize_clean(text):
    return [t for t in str(text).split() if len(t)>1]

event_counts={}
global_counts=Counter()
for ev,g in strict_df.groupby('canonical_event'):
    c=Counter()
    for txt in g['clean_text']:
        c.update(tokenize_clean(txt))
    event_counts[ev]=c; global_counts.update(c)

top_vocab=[w for w,_ in global_counts.most_common(20000)]
vocab_idx={w:i for i,w in enumerate(top_vocab)}
shift_rows=[]
for ev,c_ev in event_counts.items():
    c_train=global_counts-c_ev
    p=np.array([c_ev.get(w,0) for w in top_vocab],dtype=float)+1e-12
    q=np.array([c_train.get(w,0) for w in top_vocab],dtype=float)+1e-12
    p/=p.sum(); q/=q.sum()
    js=float(jensenshannon(p,q,base=2.0)**2)
    total=sum(c_ev.values())
    oov=sum(n for w,n in c_ev.items() if c_train.get(w,0)==0)/max(total,1)
    shift_rows.append({'canonical_event':ev,'rows':int((strict_df['canonical_event']==ev).sum()),'token_count':int(total),'lexical_js_divergence':js,'oov_token_mass':float(oov)})
shift=pd.DataFrame(shift_rows)

# Add per-event macro-F1 from saved/new OOF predictions.
base_oof=prior_event[['global_row_id','canonical_event','label_11','pred_11class__tfidf_svc','pred_11class__tfidf_lr','pred_11class__minilm_lr']].copy()
base_oof=base_oof.merge(event_oof[['global_row_id','pred_11class__distilbert_ft']],on='global_row_id',validate='one_to_one')
perf=[]
for ev,g in base_oof.groupby('canonical_event'):
    for model,col in [('tfidf_svc','pred_11class__tfidf_svc'),('tfidf_lr','pred_11class__tfidf_lr'),('minilm_lr','pred_11class__minilm_lr'),('distilbert_ft','pred_11class__distilbert_ft')]:
        perf.append({'canonical_event':ev,'model':model,'event_macro_f1':float(f1_score(g['label_11'],g[col],labels=LABELS,average='macro',zero_division=0)),'event_accuracy':float(accuracy_score(g['label_11'],g[col]))})
perf=pd.DataFrame(perf)
shift_perf=perf.merge(shift,on='canonical_event',validate='many_to_one')
shift_perf.to_csv(TABLE_DIR/'event_lexical_shift_and_performance.csv',index=False)

corr=[]
for model,g in shift_perf.groupby('model'):
    for metric in ['lexical_js_divergence','oov_token_mass']:
        rho,p=spearmanr(g[metric],g['event_macro_f1'])
        corr.append({'model':model,'shift_metric':metric,'spearman_rho_with_event_macro_f1':float(rho),'p_value_descriptive':float(p),'n_events':int(len(g))})
pd.DataFrame(corr).to_csv(TABLE_DIR/'event_shift_performance_spearman.csv',index=False)
print(pd.DataFrame(corr).to_string(index=False))


## Two-rater manual audit

The reviewer specifically questioned the mechanically derived triage proxy. The notebook therefore creates a two-rater version of the existing 200-message audit sample. **This step requires real human annotation.**

Recommended labels for each rater: `request`, `response`, `other`, or `uncertain`.

After both raters complete the CSV and save it as `triage_manual_audit_two_rater_COMPLETED.csv` in `VirconPaper1`, rerun the following analysis cell. It calculates inter-rater agreement and estimates how often the current `other` mapping contains implicit requests/responses.

In [ ]:
# 14. Prepare two-rater audit template; analyze it only if completed by humans.
from sklearn.metrics import cohen_kappa_score

prior_audit=pd.read_csv(PRIOR_TABLE/'triage_manual_audit_template.csv')
audit_cols=['global_row_id','source_split','source_row','event','label_11','label_triage','raw_text']
audit2=prior_audit[audit_cols].copy()
for c in ['rater1_triage','rater2_triage','rater1_implicit_request','rater2_implicit_request','rater1_implicit_response','rater2_implicit_response','rater1_notes','rater2_notes']:
    audit2[c]=''
template_path=WORKDIR/'triage_manual_audit_two_rater_template.csv'
if not template_path.is_file():
    audit2.to_csv(template_path,index=False)
print('Human audit template:',template_path)

completed_path=WORKDIR/'triage_manual_audit_two_rater_COMPLETED.csv'
if completed_path.is_file():
    aud=pd.read_csv(completed_path).fillna('')
    valid_labels={'request','response','other','uncertain'}
    mask=aud['rater1_triage'].isin(valid_labels)&aud['rater2_triage'].isin(valid_labels)
    if mask.sum()<50:
        print('Completed audit file found, but fewer than 50 rows have both rater labels. Finish annotation first.')
    else:
        a=aud.loc[mask,'rater1_triage']; b=aud.loc[mask,'rater2_triage']
        kappa=float(cohen_kappa_score(a,b,labels=['request','response','other','uncertain']))
        agreement=float((a==b).mean())
        consensus=np.where(a==b,a,'disagreement')
        aud.loc[mask,'consensus_triage']=consensus
        summary={
            'n_double_rated':int(mask.sum()),'raw_agreement':agreement,'cohen_kappa':kappa,
            'consensus_request_fraction':float(np.mean(consensus=='request')),
            'consensus_response_fraction':float(np.mean(consensus=='response')),
            'consensus_other_fraction':float(np.mean(consensus=='other')),
            'disagreement_fraction':float(np.mean(consensus=='disagreement')),
        }
        pd.DataFrame([summary]).to_csv(TABLE_DIR/'triage_manual_audit_agreement_summary.csv',index=False)
        aud.to_csv(TABLE_DIR/'triage_manual_audit_completed_with_consensus.csv',index=False)
        # Per source-label contamination estimate among audited items.
        sub=aud.loc[mask].copy(); sub['consensus_triage']=consensus
        per=(sub.groupby('label_11')['consensus_triage'].value_counts(normalize=True).rename('fraction').reset_index())
        per.to_csv(TABLE_DIR/'triage_manual_audit_per_source_label_sensitivity.csv',index=False)
        print(pd.DataFrame([summary]).to_string(index=False))
else:
    print('No completed human audit yet. This is expected. Fill and save:', completed_path.name)


In [ ]:
# 15. Deployment benchmark for fine-tuned DistilBERT on CPU, same text path.
# This is measured in the current runtime; use it alongside a same-runtime re-benchmark if strict hardware matching is required.
def benchmark_transformer_cpu(model_dir, texts, repeats=200, batch_n=1000):
    tok=AutoTokenizer.from_pretrained(model_dir,use_fast=True)
    model=AutoModelForSequenceClassification.from_pretrained(model_dir).to('cpu').eval()
    torch.set_num_threads(1)
    sample=list(map(str,texts))
    # warm-up
    for t in sample[:10]:
        enc=tok(t,truncation=True,padding=True,max_length=MAX_LENGTH,return_tensors='pt')
        with torch.no_grad(): _=model(**enc).logits
    times=[]
    for t in sample[:repeats]:
        enc=tok(t,truncation=True,padding=True,max_length=MAX_LENGTH,return_tensors='pt')
        t0=time.perf_counter()
        with torch.no_grad(): _=model(**enc).logits
        times.append((time.perf_counter()-t0)*1000)
    batch_text=sample[:min(batch_n,len(sample))]
    t0=time.perf_counter()
    for i in range(0,len(batch_text),EVAL_BATCH_SIZE):
        enc=tok(batch_text[i:i+EVAL_BATCH_SIZE],truncation=True,padding=True,max_length=MAX_LENGTH,return_tensors='pt')
        with torch.no_grad(): _=model(**enc).logits
    batch_ms=(time.perf_counter()-t0)*1000/max(len(batch_text),1)
    size_bytes=sum(p.stat().st_size for p in Path(model_dir).rglob('*') if p.is_file())
    return {'model':'distilbert_ft','pipeline_size_mb':size_bytes/(1024**2),'median_single_ms':float(np.median(times)),'p95_single_ms':float(np.quantile(times,.95)),'batch_ms_per_message':float(batch_ms),'device':'cpu','torch_threads':1}

bench=benchmark_transformer_cpu(fixed_model_11_dir,test_df['clean_text'].tolist())
pd.DataFrame([bench]).to_csv(TABLE_DIR/'distilbert_cpu_deployment_benchmark.csv',index=False)
print(bench)


In [ ]:
# 16. Final reviewer-facing summary and reproducibility package.
summary_files=[
'distilbert_fixed_split_summary.csv','distilbert_event_disjoint_summary.csv',
'distilbert_primary_effect_estimates_and_cis.csv','selective_prediction_abstention_curves.csv',
'event_lexical_shift_and_performance.csv','event_shift_performance_spearman.csv','distilbert_cpu_deployment_benchmark.csv']
if RUN_EXTERNAL_HUMAID:
    summary_files += ['humaid_external_transfer_summary.csv','humaid_external_eventwise_metrics.csv','humaid_exact_overlap_by_event.csv']

missing=[f for f in summary_files if not (TABLE_DIR/f).is_file()]
if missing:
    raise RuntimeError('Continuation incomplete. Missing outputs: '+', '.join(missing))

# Environment + source hashes.
env={
    'python':sys.version,'platform':platform.platform(),'numpy':np.__version__,'pandas':pd.__version__,
    'scipy':scipy.__version__,'sklearn':sklearn.__version__,'torch':torch.__version__,
    'cuda_available':torch.cuda.is_available(),'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
(REPORT_DIR/'environment.json').write_text(json.dumps(env,indent=2),encoding='utf-8')

def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as fh:
        for chunk in iter(lambda:fh.read(1024*1024),b''): h.update(chunk)
    return h.hexdigest()
input_hashes={p.name:sha256(p) for p in (TRAIN_PATH,TEST_PATH,PRIOR_ZIP)}
(REPORT_DIR/'input_hashes.json').write_text(json.dumps(input_hashes,indent=2),encoding='utf-8')

claim_note={
'fixed_split':'Official CrisisBench split; do not describe as unseen-disaster generalization.',
'event_disjoint':'Same-corpus event shift on 50 canonical events; stronger than message-level split but not an independent corpus.',
'external_humaid':'External transfer result on HumAID test events after exact normalized-text overlap removal. Multi-class results use the explicit shared ontology only.',
'abstention':'Selective prediction is a computational confidence analysis, not a measured human workload/queue-reduction study.',
'manual_audit':'Only claim practitioner/manual validation after the two-rater file is actually completed.'}
(REPORT_DIR/'claim_scope_and_limitations_continuation.json').write_text(json.dumps(claim_note,indent=2),encoding='utf-8')

# Artifact checksums.
manifest=[]
for p in sorted(OUT_ROOT.rglob('*')):
    if p.is_file(): manifest.append({'relative_path':str(p.relative_to(OUT_ROOT)),'sha256':sha256(p),'bytes':p.stat().st_size})
pd.DataFrame(manifest).to_csv(OUT_ROOT/'artifact_manifest_sha256.csv',index=False)

zip_path=WORKDIR/'VIRCON_IJIES_Continuation_Strengthening_Artifacts.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED,allowZip64=True) as z:
    for p in OUT_ROOT.rglob('*'):
        if p.is_file(): z.write(p,arcname=str(p.relative_to(OUT_ROOT.parent)))
print('\nCONTINUATION COMPLETED')
print('Artifact directory:',OUT_ROOT)
print('Final continuation ZIP:',zip_path)
print('Two-rater audit template:',template_path)


## What to upload back here

After the full run completes, upload:

1. `VIRCON_IJIES_Continuation_Strengthening_Artifacts.zip`
2. **If completed by two human raters:** `triage_manual_audit_two_rater_COMPLETED.csv`

I can then merge these results with the earlier reviewer-rerun outputs, decide which new analyses belong in the main paper versus supplementary material, update the manuscript with minimal disruption to the humanized wording, and prepare the point-by-point IJIES response.